In [0]:
%run "/Workspace/Local To databrick manish migration/Resources/Dev/Con_note"


In [0]:
%run "/Workspace/Local To databrick manish migration/src/utility/s3_client_object"

In [0]:
%run "/Workspace/Local To databrick manish migration/src/utility/encrypt_decrypt"

In [0]:
%run "/Workspace/Local To databrick manish migration/src/utility/logging_config"

In [0]:
%run "/Workspace/Local To databrick manish migration/src/utility/move_files"

In [0]:
### get s3 client ###


# s3_client_provider = S3ClientProvider(decrypt(aws_access_key), decrypt(aws_secret_key))
# s3_client = s3_client_provider.get_client()
# spark = spark_session()

# spark.conf.set("fs.s3a.access.key", decrypt(aws_access_key))
# spark.conf.set("fs.s3a.secret.key", decrypt(aws_secret_key))
# spark.conf.set("fs.s3a.endpoint", "s3.ap-southeast-2.amazonaws.com")
# spark.conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")


#### Now you can use s3 Client ####
# response = s3_client.list_buckets()
# print (response)
#logger.info("List of Buckets: %s", response['Buckets'])


# check if local directory has already a file
# if file is there then check if the same file is present in the staging area
# with status as A. If so then don't delete and try to re-run
# Else give an error and not process the next file
# csv_files = [file for file in os.listdir(config. local_directory) if file. endswith(".csv")]



df = spark.read.format("binaryFile").load("s3://de-manish-project/sales_data/")

all_files = [row.path for row in df.select("path").collect()]

logger.info(f"List of Buckets: {all_files}")

logger.info("******************Fetching CSV Files Only:************************")

csv_files = []
error_files = []

if all_files:
    for file in all_files:   # ✅ fixed variable name
    

        if file.endswith(".csv"):
            csv_files.append(file)   # ✅ only 1 argument
        else:
            error_files.append(file)

    logger.info(f"CSV Files are: {csv_files}")
    logger.info(f"Error Files are: {error_files}")

else:
    logger.error("There is no data to Process")
    raise Exception("There is no data to process.")

# total_csv_files = []
# if csv_files:
#     for file in csv_files:
#         file_name = file.split("/")[-1]  # ✅ extract filename
#         total_csv_files.append(f"'{file_name}'")

#     print(total_csv_files)

#     data = spark.sql(f"""
#             SELECT DISTINCT file_name
#             FROM product_staging_table
#             WHERE file_name IN ({','.join(total_csv_files)})
#             AND status = 'F'
#         """)

#     # logger.info(f"List of files that are still in progress:- {data.display()}")

#     if data:
#         logger.info("Your last run was failed please check")
#     else:
#         logger.info("No Data Match")
# else:
#     logger.info("Last run was successful !!! ")

# logger.info(f"List of csv files that needs to be processed s {csv_files}")

total_csv_files=[]
if csv_files:
    for f in csv_files:
        name=f.split("/")[-1]
        total_csv_files.append(f"'{name}'")
    try:
        data = spark.sql(f"""select distinct file_name,status
                    from workspace.default.product_staging_table
                    where file_name in ({','.join(total_csv_files)})  AND status = 'File In Progress'
                    """)
    except Exception as e:
        logger.error(f"Error in staging {e}")
    # data = spark.sql(f"""select distinct File_Name
    #                  from main_optimization_table.main_tables.product_staging_table
    #                  where File_Name in ({','.join(files_names)})  AND status = 'File In Progress'
    #                  """)
    data.show()
    if data.count() > 0:
        logger.info("Please check last run")
    else:
        logger.info("No file in progress")
else:
    logger.info("No CSV file to process")









In [0]:
correct_files = []
for data in csv_files:
    data_schema = spark.read.format("csv")\
        .option("header", "true")\
        .load(data).columns
    logger.info(f"Schema for the {data} is {data_schema}")
    logger.info(f"Mandatory columns schema is {mandatory_columns}")
    missing_columns = set(mandatory_columns) - set(data_schema)
    logger.info(f"missing columns are {missing_columns}")

    if missing_columns:
        error_files.append(data)
    else:
        logger.info(f"No missing column for the {data} ")
        correct_files.append(data)

logger.info(f" *********** List of correct files after checking schema *************** {correct_files}")
logger.info(f" *********** List of error files checking schema*************** {error_files}")


In [0]:
error_files

In [0]:
logger.info("******* Moving Error data to error directory if any ************")

for file in error_files:
    file_name = file.split("/")[-1]

    message = move_file_s3(
        file,
        f"s3://de-manish-project/sales_data_error/{file_name}"
    )

    logger.info(message)

logger.info("******* Moved successfully ************")

In [0]:
import datetime


In [0]:
correct_files

In [0]:
logger.info(f" ********Updating the product_staging_table that we have started the process ***")
#


from datetime import datetime
import pytz

ist = pytz.timezone('Asia/Kolkata')

current_date = datetime.now(ist)

formatted_date = current_date.strftime("%Y-%m-%d %H:%M:%S")

print(formatted_date)







insert_statements = []
# current_date = datetime.datetime.now()
# formatted_date = current_date.strftime("%Y-%m-%d %H:%M:%S")
id=0
if correct_files:
    for file in correct_files:
        file_name = file.split("/")[-1]  # ✅ extract filename
        # statements= (f"INSERT INTO {db_name}. {config.product_staging_table}"
        #              f"(file_name, file_location, created_date, status)")

        statements = spark.sql(f"""
                    INSERT INTO workspace.default.product_staging_table
                      (id,file_name, file_location, created_date, status)
                      VALUES ('{id}','{file_name}', '{file}','{formatted_date}' ,'File In Progress')
        """)
        id+=1
        insert_statements.append(statements)

    logger.info(f"Insert statement created for staging table --- {insert_statements}")
else:
    logger.error(" ********** There is no files to process ************")
    raise Exception(" ************ No Data avalable with correct files ****")

logger.info(" ****************** Staging table updated successfully *************")

In [0]:
%sql
select * from product_staging_table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
logger.info("**************** Fixing extra column coming from source ********************************")

schema = StructType([
        StructField("customer_id", IntegerType(), True),
        StructField("store_id", IntegerType(), True),
        StructField("product_name", StringType(), True),
        StructField("sales_date", DateType(), True),
        StructField("sales_person_id", IntegerType(), True),
        StructField("price", FloatType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("total_cost", FloatType(), True),
        StructField("additional_column", StringType(), True)
])
#
#connecting with DatabaseReader
# database_client = DatabaseReader(config.url, config.properties)
# logger.info(" ************** creating empty dataframe ******")
# final_df_to_process = database_client.create_dataframe(spark,"empty_df_create_table")
#
final_df_to_process = spark.createDataFrame([], schema=schema)
# # Create a new column with concatenated values of extra columns
#
for data in correct_files:
    data_df = spark.read. format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(data)
    data_schema = data_df.columns
    extra_columns = list(set(data_schema) - set(mandatory_columns))
    logger.info(f"Extra columns present at source is {extra_columns}")
    if extra_columns:
        data_df = data_df.withColumn("additional_column", concat_ws(", ", *extra_columns)) \
                .select("customer_id", "store_id", "product_name", "sales_date", "sales_person_id", "price", "quantity", "total_cost", "additional_column")
        logger.info(f"processed {data} and added 'additional_column'")
    else:
        data_df = data_df.withColumn("additional_column", lit(None)) \
        .select("customer_id", "store_id", "product_name", "sales_date", "sales_person_id","price", "quantity", "total_cost", "additional_column")

    final_df_to_process = final_df_to_process.union(data_df)
# final_df_to_process = data_df
    logger.info(" ********* Final Dataframe from source which will be going****")

logger.info(" ********* Final Dataframe from source which will be going to processing *******")
final_df_to_process.show(25)
# final_df_to_process.coalesce(1).write.format("csv")\
#     .option('header',True)\
#     .mode("append")\
#     .save('E:\PY_charm_Manish\spark_data')
print(final_df_to_process.count())

In [0]:
logger.info(" ************** Loading customer table into customer_table_df***********")
customer_table_df = spark.sql("select * from youtube_project.youtube_project_database.customer")
#product table
logger.info(" ************* Loading product table into product_table_df *****")
product_table_df = spark.sql("select * from youtube_project.youtube_project_database.product")

#product_staging_table table
logger.info(" ************** Loading satging table into product_staging_table_df****")
product_staging_table_df = spark.sql("select * from youtube_project.youtube_project_database.customer")


#sales_team table
logger.info(" ************** Loading sales team table into sales_team_table_df********")
sales_team_table_df = spark.sql("select * from youtube_project.youtube_project_database.sales_team")

#store table
logger.info(" ************** Loading store table into store_table_df *********")
store_table_df = spark.sql("select * from youtube_project.youtube_project_database.store")

In [0]:
%run "/Workspace/Local To databrick manish migration/src/main/transformation/jobs/dimension_tables_join"

In [0]:

s3_customer_store_sales_df_join = dimesions_table_join(final_df_to_process,
                                                       customer_table_df,
                                                       store_table_df,
                                                       sales_team_table_df)
#Final enriched data
logger.info(" ************ Final Enriched Data***********")
s3_customer_store_sales_df_join.display()

In [0]:
logger.info(" *************** write the data into Customer Data Mart ********** ")
final_customer_data_mart_df = s3_customer_store_sales_df_join\
    .select(
    "ct.customer_id",
    "customer_first_name",
    "customer_last_name",
    "customer_address",
    "customer_pincode",
    "phone_number",
    "sales_date",
    "total_cost"
)
logger.info(" ************* Final Data for customer Data Mart ***********")
final_customer_data_mart_df.show()
final_customer_data_mart_df.display()

logger.info(" ************* Started writing customer Data Mart to s3a://de-manish-project/customer_data_mart/ ***********")

# we can write this in parquet formate as well
# final_customer_data_mart_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save("s3://de-manish-project/customer_data_mart/")

final_customer_data_mart_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .save("s3a://de-manish-project/customer_data_mart/")
logger.info(" ************* success fully wrote to s3a://de-manish-project/customer_data_mart/ ***********")


In [0]:
logger.info(" *************** write the data into sales team Data Mart ********** ")
final_sales_team_data_mart_df = s3_customer_store_sales_df_join\
    .select("store_id","sales_person_id","sales_person_first_name" ,"sales_person_last_name",
            "store_manager_name", "manager_id", "is_manager","sales_person_address",
            "sales_person_pincode","sales_date", "total_cost",
            expr("SUBSTRING(sales_date,1,7) as sales_month"))


logger.info(" ************* Final Data for sales_team Data Mart ***********")
final_sales_team_data_mart_df.show()
final_sales_team_data_mart_df.display()

logger.info(" ************* Started writing sales team Data Mart s3a://de-manish-project/sales_data_mart/ ***********")

final_sales_team_data_mart_df.write.format("delta")\
    .mode("overwrite")\
    .save('s3a://de-manish-project/sales_data_mart/')

logger.info(f" ************* Successfully wrote sales team Data Mart to local Dosk at ***** s3a://de-manish-project/sales_data_mart/")


In [0]:
logger.info("*** Started writing sales team Data Mart partitionBy(sales_month,store_id) s3a://de-manish-project/sales_data_mart/ ***********")
final_sales_team_data_mart_df.write.format('delta')\
    .mode('overwrite')\
    .partitionBy("sales_month","store_id")\
    .save('s3a://de-manish-project/sales_partitioned_data_mart')

logger.info("*** Successfully wrote sales team Data Mart partitionBy(sales_month,store_id) s3a://de-manish-project/sales_data_mart/ ***********")


In [0]:
%run "/Workspace/Local To databrick manish migration/src/main/transformation/jobs/customer_mart_sql_tranform_write"

In [0]:
logger.info(" ****** Calculating customer every month purchased amount ******* ")

customer_mart_calculation_table_write(final_customer_data_mart_df)

logger.info(" ****** Calculation of customer mart done and written into the table ********* ")

In [0]:
%sql
select * from youtube_project.youtube_project_database.customers_data_mart

In [0]:
%run "/Workspace/Local To databrick manish migration/src/main/transformation/jobs/sales_mart_sql_transform_write"

In [0]:
final_sales_team_data_mart_df.printSchema()

In [0]:
logger.info(" ****** Calculating sales every month billed amount ******* ")

sales_mart_calculation_table_write(final_sales_team_data_mart_df)

logger.info(" ****** Calculation of sales mart done and written into the table ******** ")

In [0]:
%sql
select * from youtube_project.youtube_project_database.sales_team_data_mart

In [0]:
logger.info(" ****** Checking Files After processed   ******* ")
final_file_check = spark.read.format("binaryFile").load("s3://de-manish-project/sales_data/")

final_file = [row.path for row in final_file_check.select("path").collect()]

logger.info(f"List of Buckets: {final_file}")


In [0]:
logger.info("******* Moving correct data to processed directory ************")

for file in correct_files:
    file_name = file.split("/")[-1]

    message = move_file_s3(
        file,
        f"s3://de-manish-project/sales_data_processed/{file_name}"
    )

    logger.info(message)

logger.info("******* Moved successfully ************")

In [0]:
update_statements = []




from datetime import datetime
import pytz

ist = pytz.timezone('Asia/Kolkata')

current_date = datetime.now(ist)

formatted_date = current_date.strftime("%Y-%m-%d %H:%M:%S")

print(formatted_date)

# current_date_ = datetime.datetime.now()
# formatted_date_ = current_date_.strftime("%Y-%m-%d %H:%M:%S")
if correct_files:
    for file in correct_files:
        file_name = file.split("/")[-1]

        spark.sql(f"""
    UPDATE workspace.default.product_staging_table
    SET status = 'Done', updated_date = '{formatted_date}'
    WHERE file_name = '{file_name}'
""")
        # statements = f"UPDATE {db_name}. {config.product_staging_table} " \
        #              f" SET status = 'Done', updated_date='{formatted_date_}' " \
        #              f"WHERE file_name = '{file_name}'"

        update_statements.append(statements)
    logger.info(f"Updated statement created for staging table --- {update_statements}")

In [0]:
%sql
select * from workspace.default.product_staging_table

In [0]:
print(update_statements)

In [0]:
correct_files

In [0]:
# dbutils.fs.rm("s3a://de-manish-project/sales_data_mart/", True)

In [0]:
# final_customer_data_mart_df.show()

In [0]:
# final_customer_data_mart_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save("s3a://de-manish-project/customer_data_mart/")

In [0]:
# print(move_file_s3.__code__.co_varnames)

In [0]:
# %sql
# CREATE TABLE product_staging_table (
#     id BIGINT,
#     file_name STRING,
#     file_location STRING,
#     created_date TIMESTAMP,
#     updated_date TIMESTAMP,
#     status STRING
# )
# USING DELTA;

In [0]:
# %sql
# INSERT INTO product_staging_table VALUES
# (1, 'file1.csv', 's3://de-manish-project/sales_data/file1.csv', current_timestamp(), current_timestamp(), 'I'),
# (2, 'file2.csv', 's3://de-manish-project/sales_data/file2.csv', current_timestamp(), current_timestamp(), 'C');

In [0]:
%sql
select * from product_staging_table

In [0]:
%sql
-- truncate table product_staging_table

In [0]:
# %sql
# INSERT INTO product_staging_table VALUES
# (4, 'sales_data.csv', 's3://de-manish-project/sales_data/sales_data.csv', current_timestamp(), current_timestamp(), 'F')